In [1]:
import pandas as pd
from gprofiler import GProfiler
gp = GProfiler(return_dataframe=True)
from enum import Enum
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
class canEnum(Enum):
    HNSCC = 0
    LSCC = 1
    CCRCC = 2
    LUAD = 3
    EN = 4

In [3]:
delta_correlation_df = pd.read_csv('data/delta_correlation_df_with_significance.csv')
delta_correlation_df

,Gene,Delta_Correlation,P_Value,FDR,Cancer,Significant
0,A1BG,-0.198013,2.451044e-01,4.045115e-01,HNSCC,False
1,A2M,-0.118384,4.480278e-01,6.091130e-01,HNSCC,False
2,A2ML1,-0.023469,2.918125e-01,4.561968e-01,HNSCC,False
3,AAAS,0.275905,1.051756e-01,2.209072e-01,HNSCC,False
4,AACS,-0.136836,1.800586e-01,3.266475e-01,HNSCC,False
...,...,...,...,...,...,...
50684,ZWINT,1.219024,2.267627e-09,1.049863e-07,Endometrial,True
50685,ZXDC,-0.346532,2.983295e-01,5.386144e-01,Endometrial,False
50686,ZYG11B,0.768196,5.463938e-04,5.319699e-03,Endometrial,True
50687,ZYX,0.253630,2.456049e-01,4.795301e-01,Endometrial,False


In [4]:
kegg_profiles = []
for cancer in pd.unique(delta_correlation_df.Cancer):
    
    cancer_df = delta_correlation_df[delta_correlation_df.Cancer == cancer]
    background_genes  = list(pd.unique(cancer_df.Gene))
    cancer_df = cancer_df[cancer_df.Significant == True]
    queryS = list(pd.unique(cancer_df.Gene))
    cancer_profile = gp.profile(organism='hsapiens', query = queryS, no_iea=True, sources = ["KEGG"],
                       ordered=True, no_evidences=False, background= background_genes)
    kegg_profiles.append(cancer_profile)

In [5]:
endo = kegg_profiles[canEnum.EN.value][['description','intersections']]
luad = kegg_profiles[canEnum.LUAD.value][['description','intersections']]
lscc = kegg_profiles[canEnum.LSCC.value][['description','intersections']]
hnscc = kegg_profiles[canEnum.HNSCC.value][['description','intersections']]
ccrcc = kegg_profiles[canEnum.CCRCC.value][['description','intersections']]

In [6]:
def create_gene_cancer_dataframe(gene_lists, cancer_names):

    all_genes_set = set()
    for sublist in gene_lists:
        for gene in sublist:
            all_genes_set.add(gene)
    

    unique_genes_sorted = sorted(list(all_genes_set))

    data_for_df = {}

    for i, cancer_name in enumerate(cancer_names):
        current_cancer_genes = set(gene_lists[i]) 
        gene_presence_column = []
        for gene in unique_genes_sorted:
            if gene in current_cancer_genes:
                gene_presence_column.append(1)
            else:
                gene_presence_column.append(0)
        data_for_df[cancer_name] = gene_presence_column

    df = pd.DataFrame(data_for_df, index=unique_genes_sorted)

    num_cancers = len(cancer_names)
    if num_cancers > 0:
        df['Total'] = df.sum(axis=1)


    return df


In [7]:

pathways = ['Valine, leucine and isoleucine degradation', 
    'Fatty acid metabolism', 'p53 signaling pathway', 
    'Propanoate metabolism', 'beta-Alanine metabolism',
    'Glycolysis / Gluconeogenesis',
    'Butanoate metabolism', 'Glycerolipid metabolism']

interaction_dfs = []

for pathway in pathways:
    intersect_list = []
    cancer_list = []
    for i, cancer in zip([endo, luad, lscc, hnscc, ccrcc],["Endo", "LUAD", "LSCC", "HNSCC", "CCRCC"]):
        if pathway in i.description.values:
            intersect_list.append(list(i[i.description == pathway].intersections)[0])
            cancer_list.append(cancer)
        else:
            cancer_list.append(cancer)
            intersect_list.append([])

    intersect = set(intersect_list[0])
    for i in intersect_list[1:5]:
        intersect.intersection_update(i)
    pathway_df = create_gene_cancer_dataframe(intersect_list, cancer_list)
    interaction_dfs.append(pathway_df)
    # plot_gene_representation(pathway_df, pathway)


In [8]:

pathways = ['Valine, leucine and isoleucine degradation', 
    'Fatty acid metabolism', 'p53 signaling pathway', 
    'Propanoate metabolism', 'beta-Alanine metabolism',
    'Glycolysis or Gluconeogenesis',
    'Butanoate metabolism', 'Glycerolipid metabolism']

with pd.ExcelWriter("Figures/Table_S1_Pathway_Genes.xlsx",
    mode='w',
    engine="openpyxl",) as writer:
    for table, pathway in zip(interaction_dfs, pathways):
        table.to_excel(writer, sheet_name=pathway)

/Users/jhgirald/mambaforge/envs/proteinmrna/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
